In [14]:
import os
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Load environment variables first
load_dotenv()
parser = StrOutputParser()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
URL = os.getenv("URL")
API_KEY = os.getenv("APIKEY")


In [15]:
# 2. Initialize Models
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0,api_key=GROQ_API_KEY)

In [16]:
embeddings = HuggingFaceEmbeddings(model_name="BAAI/bge-small-en-v1.5")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3478.11it/s]


In [17]:
# 3. File Loading
txtfile_path = "yarvalley.txt"
loader = TextLoader(txtfile_path)
txtfile = loader.load()

# 4. Text Splitting
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=[
        "\n\n",   # paragraphs (highest priority)
        "\n",     # lines
        ". ",     # sentences
        " ",      # words
        ""        # fallback
    ]
)
split_document = splitter.split_documents(txtfile)

# 5. Vector Store Ingestion
vectorstore = QdrantVectorStore.from_documents(
    documents=split_document,
    embedding=embeddings,
    api_key=API_KEY,
    url=URL,
    collection_name="vanilla_rag",
)

In [18]:
retriever = vectorstore.as_retriever(search_kwargs={"k":5})

rag_prompt = ChatPromptTemplate.from_template(
    """
    Please answer the following questions,if it is in context otherwise answer "I don,t have enough information about this"
    Context:
    {context}

    Question:
    {question}


    """
)

In [19]:
#This return objects in a string format
def get_content(docs):
    return "\n\n".join( doc.page_content for doc in docs )

In [20]:
#context and question will fill this(rag_prompt)
#LCEL

rag_chain = (
    {
        "context": retriever | get_content,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | parser)

In [21]:
question = "Where is malam jabba in swat"
answer = rag_chain.invoke(question)

print(answer)

Malam Jabba is located about 44 km from Mingora in Swat.


**RAGAS(RAG Assesment)**

In [22]:
test_questions = [
    "Where is Swat?",
    "Where is Malam Jabba in Swat?",
    "What is Swat famous for?",
    "How many tourist spots are there in Swat?",
]

ground_truths = [
    # Q1
    "Swat Valley is located in the Malakand Division of Khyber Pakhtunkhwa province of Pakistan, situated north of Peshawar between 34°40' to 35°N latitude and 72° to 74°6'E longitude.",

    # Q2
    "Malam Jabba is located about 44 km from Mingora. It is a modern hill resort featuring snowy mountain peaks, green valleys, forests, a chairlift, and a ski resort restored by TCKP in 2015.",

    # Q3
    "Swat is famous for its scenic natural beauty earning it the title Switzerland of the East, Buddhist civilization remnants and Gandhara art, emerald mines near Mingora, Malam Jabba ski resort, and tourist spots like Kalam, Bahrain, Madyan, Marghuzar, and Miandam.",

    # Q4
    "Swat has several tourist spots including Malam Jabba, Kalam, Bahrain, Madyan, Miandam, Marghuzar, Bishigram Valley, Mankial Valley, Mingora bazaar, Saidu Sharif, and Swat Museum.",
]

In [29]:
answers = []
contexts = []

for question in test_questions:
    # get answer from LLM
    answer = rag_chain.invoke(question)
    answers.append(answer)

    # get contexts from Qdrant
    retrieved_docs = retriever.invoke(question) #Dear retriever goto qdrant store and search against this question and give me content for this question
    retrieved_docs_content = [doc.page_content for doc in retrieved_docs]
    contexts.append(retrieved_docs_content)